In [23]:
from pathlib import Path
import re
import pandas as pd

ROMULUS = Path('/mnt/data0/pkrsnak/romulus')
RUNS = ROMULUS / 'runs'

run_dirs = sorted(p for p in RUNS.iterdir() if p.is_dir())
len(run_dirs)

52

In [17]:
def parse_key(key):
    fam, _, variant = key.partition('_')     # 'r488_BH6' -> ('r488', '_', 'BH6'); 'r107' -> ('r107', '', '')
    return fam, variant or '?'

parse_key('r488_BH6'), parse_key('r107')

(('r488', 'BH6'), ('r107', '?'))

In [24]:
[h.name for h in (RUNS / 'r320').iterdir()]

['r320.007779.tipsy']

In [ ]:
DUMP_2026_09 = ['r431', 'r489', 'r502', 'r515', 'r556', 'r563', 'r571']

def lineage(p):
    if p.is_symlink():
        return 'enterprise'
    if any(p.glob('*.tipsy')):
        return 'tal-shiar-15'
    fam, _ = parse_key(p.name)
    if fam in DUMP_2026_09:
        return 'dump-2026-09'
    return 'unknown'

{name: lineage(RUNS / name) for name in ['r488_BH', 'r107', 'r431_noBH']}

{'r488_BH': 'enterprise', 'r107': 'tal-shiar-15', 'r431_noBH': 'dump-2026-09'}

In [26]:
STEP_RE = re.compile(r'\.(\d{6})(\.tipsy)?$')    # matches '.004096' or '.007779.tipsy' at the end of a name

def find_snapshot(p):
    hits = [h for h in p.iterdir() if STEP_RE.search(h.name)]
    assert len(hits) == 1, (p.name, [h.name for h in hits])
    h = hits[0]
    step = STEP_RE.search(h.name).group(1)
    if h.is_dir():
        return step, 'stepdir', h / h.name    # dump: the file sits inside a folder of the same name
    return step, 'flat', h                    # enterprise and the 15: the file is right there

find_snapshot(RUNS / 'r488_BH'), find_snapshot(RUNS / 'r431_noBH'), find_snapshot(RUNS / 'r107')

(('004096',
  'flat',
  PosixPath('/mnt/data0/pkrsnak/romulus/runs/r488_BH/r488.romulus25.3072g1HsbBH.004096')),
 ('004096',
  'stepdir',
  PosixPath('/mnt/data0/pkrsnak/romulus/runs/r431_noBH/r431.romulus25.3072g1Hsb.004096/r431.romulus25.3072g1Hsb.004096')),
 ('007779',
  'flat',
  PosixPath('/mnt/data0/pkrsnak/romulus/runs/r107/r107.007779.tipsy')))

In [27]:
rows = []
for p in run_dirs:
    fam, variant = parse_key(p.name)
    step, layout, snap = find_snapshot(p)
    rows.append(dict(
        key=p.name, family=fam, variant=variant, lineage=lineage(p),
        changa_name=STEP_RE.sub('', snap.name),
        step=step, layout=layout,
        real_path=str(p.resolve()),
        owner=p.resolve().owner(),
    ))

df = pd.DataFrame(rows)
df

,key,family,variant,lineage,changa_name,step,layout,real_path,owner
0,r107,r107,?,tal-shiar-15,r107,007779,flat,/mnt/data0/pkrsnak/romulus/runs/r107,pkrsnak
1,r142,r142,?,tal-shiar-15,r142,007779,flat,/mnt/data0/pkrsnak/romulus/runs/r142,pkrsnak
2,r154,r154,?,tal-shiar-15,r154,007779,flat,/mnt/data0/pkrsnak/romulus/runs/r154,pkrsnak
3,r168,r168,?,tal-shiar-15,r168,007779,flat,/mnt/data0/pkrsnak/romulus/runs/r168,pkrsnak
4,r204,r204,?,tal-shiar-15,r204,007779,flat,/mnt/data0/pkrsnak/romulus/runs/r204,pkrsnak
5,r219,r219,?,tal-shiar-15,r219,007779,flat,/mnt/data0/pkrsnak/romulus/runs/r219,pkrsnak
6,r223,r223,?,tal-shiar-15,r223,007779,flat,/mnt/data0/pkrsnak/romulus/runs/r223,pkrsnak
7,r239,r239,?,tal-shiar-15,r239,007779,flat,/mnt/data0/pkrsnak/romulus/runs/r239,pkrsnak
8,r284,r284,?,tal-shiar-15,r284,007779,flat,/mnt/data0/pkrsnak/romulus/runs/r284,pkrsnak
9,r306,r306,?,tal-shiar-15,r306,007779,flat,/mnt/data0/pkrsnak/romulus/runs/r306,pkrsnak


In [28]:
def read_param(p):
    files = list(p.glob('*.param'))
    if len(files) != 1:                 # the 15 have none; a run with two would be its own mystery
        return {}
    params = {}
    for line in files[0].read_text().splitlines():
        line = line.split('#', 1)[0].strip()      # drop comments, then whitespace
        if '=' in line:
            k, v = line.split('=', 1)             # split on the FIRST '=' only
            params[k.strip()] = v.strip()
    return params

KEEP = ['iBinaryOutput', 'dBHSinkFeedbackEff', 'dBHSinkColdDen', 'dEtaDiffusion']
params = read_param(RUNS / 'r488_BH6')
len(params), {k: params.get(k) for k in KEEP}

(101,
 {'iBinaryOutput': '1',
  'dBHSinkFeedbackEff': '0.05',
  'dBHSinkColdDen': '0.2',
  'dEtaDiffusion': '0.3'})

In [29]:
param_rows = [read_param(p) for p in run_dirs]     # read each file once
for col in KEEP:
    df[col] = [pr.get(col, '') for pr in param_rows]  # '' where the file or key is missing

df[['key', 'lineage', 'variant'] + KEEP]

,key,lineage,variant,iBinaryOutput,dBHSinkFeedbackEff,dBHSinkColdDen,dEtaDiffusion
0,r107,tal-shiar-15,?,,,,
1,r142,tal-shiar-15,?,,,,
2,r154,tal-shiar-15,?,,,,
3,r168,tal-shiar-15,?,,,,
4,r204,tal-shiar-15,?,,,,
5,r219,tal-shiar-15,?,,,,
6,r223,tal-shiar-15,?,,,,
7,r239,tal-shiar-15,?,,,,
8,r284,tal-shiar-15,?,,,,
9,r306,tal-shiar-15,?,,,,


In [30]:
df.groupby(['lineage', 'variant'])[KEEP].agg(lambda s: sorted(set(s)))

iBinaryOutput dBHSinkFeedbackEff dBHSinkColdDen  \
lineage      variant                                                   
dump-2026-09 BH             [0, 1]            [0.005]         [100.]   
             BH6            [0, 1]             [0.05]          [0.2]   
             noBH           [0, 1]                 []             []   
enterprise   BH             [0, 1]            [0.005]         [100.]   
             BH6            [0, 1]             [0.05]          [0.2]   
             BH8            [0, 1]            [0.005]          [0.2]   
             noBH           [0, 1]                 []             []   
tal-shiar-15 ?                  []                 []             []   

                         dEtaDiffusion  
lineage      variant                    
dump-2026-09 BH            [0.15, 0.3]  
             BH6           [0.15, 0.3]  
             noBH          [0.15, 0.3]  
enterprise   BH            [0.15, 0.2]  
             BH6      [0.15, 0.2, 0.3]  
             BH8      [0.15, 0.2, 0.3]  
             noBH     [0.15, 0.2, 0.3]  
tal-shiar-15 ?                      []

In [31]:
df.pivot(index='family', columns='variant', values='dEtaDiffusion')

variant,?,BH,BH6,BH8,noBH
family,,,,,
r107,,NaN,NaN,NaN,NaN
r142,,NaN,NaN,NaN,NaN
r154,,NaN,NaN,NaN,NaN
r168,,NaN,NaN,NaN,NaN
r204,,NaN,NaN,NaN,NaN
r219,,NaN,NaN,NaN,NaN
r223,,NaN,NaN,NaN,NaN
r239,,NaN,NaN,NaN,NaN
r284,,NaN,NaN,NaN,NaN


In [33]:
import os

def aux_format(p):
    iords = list(p.rglob('*.iord'))
    if not iords:
        return 'none'
    if not os.access(iords[0], os.R_OK):       # can I read it? without trying and crashing
        return 'unreadable'
    head = iords[0].read_bytes()[:16]
    return 'ascii' if all(32 <= b < 127 or b in (10, 13) for b in head) else 'binary'

def unreadable_files(p):
    return [f.relative_to(p).as_posix() for f in p.rglob('*') if f.is_file() and not os.access(f, os.R_OK)]

{p.name: unreadable_files(p) for p in run_dirs if unreadable_files(p)}

{'r488_BH8': ['r488.romulus25.3072g1HsbBH8.004096.iord'],
 'r613_BH8': ['r613.romulus25.3072g1HsbBH8.004096.iord'],
 'r613_noBH': ['r613.romulus25.3072g1Hsb.004096.iord']}

In [34]:
df['aux_format'] = [aux_format(p) for p in run_dirs]
df['unreadable'] = ['; '.join(unreadable_files(p)) for p in run_dirs]

df['notes'] = ''
df.loc[df.lineage == 'tal-shiar-15', 'notes'] += 'single tipsy at 007779, no aux/param; variant unknown, ask Jillian; '
mixed_eta = df.groupby('family')['dEtaDiffusion'].transform('nunique') > 1
df.loc[mixed_eta, 'notes'] += 'dEtaDiffusion differs across this family; '

df.to_csv(RUNS / 'MANIFEST.csv', index=False)
df[['key', 'aux_format', 'unreadable', 'dEtaDiffusion', 'notes']]

,key,aux_format,unreadable,dEtaDiffusion,notes
0,r107,none,,,"single tipsy at 007779, no aux/param; variant ..."
1,r142,none,,,"single tipsy at 007779, no aux/param; variant ..."
2,r154,none,,,"single tipsy at 007779, no aux/param; variant ..."
3,r168,none,,,"single tipsy at 007779, no aux/param; variant ..."
4,r204,none,,,"single tipsy at 007779, no aux/param; variant ..."
5,r219,none,,,"single tipsy at 007779, no aux/param; variant ..."
6,r223,none,,,"single tipsy at 007779, no aux/param; variant ..."
7,r239,none,,,"single tipsy at 007779, no aux/param; variant ..."
8,r284,none,,,"single tipsy at 007779, no aux/param; variant ..."
9,r306,none,,,"single tipsy at 007779, no aux/param; variant ..."


In [35]:
import os, grp
[grp.getgrgid(g).gr_name for g in os.getgroups()]

['pkrsnak', 'BHDG']

In [37]:
import pynbody

key = 'r431_BH'
step, layout, snap = find_snapshot(RUNS / key)
s = pynbody.load(str(snap))
h = s.halos()
len(s), len(s.gas), len(s.star), len(s.dm), len(h)

(25953621, 6714790, 2300806, 16938025, 1072)

In [41]:
import pynbody

key = 'r488_BH'
step, layout, snap = find_snapshot(RUNS / key)
s = pynbody.load(str(snap))
h = s.halos()
len(s), len(s.gas), len(s.star), len(s.dm), len(h)
s.properties['a']

0.9999999999859445

In [40]:
import pynbody

key = 'r107'
step, layout, snap = find_snapshot(RUNS / key)
s = pynbody.load(str(snap))
s.physical_units()
len(s), s['pos'].units, s.properties['a'], s.properties['boxsize']

(468580, Unit("kpc"), 0.9524351996399713, Unit("2.38e+04 kpc"))

In [42]:
def scale_factor(p):
    _, _, snap = find_snapshot(p)
    return pynbody.load(str(snap)).properties['a']

df['scale_factor'] = [round(scale_factor(p), 4) for p in run_dirs]
df.to_csv(RUNS / 'MANIFEST.csv', index=False)
df.groupby('lineage')['scale_factor'].unique()

lineage
dump-2026-09       [1.0]
enterprise         [1.0]
tal-shiar-15    [0.9524]
Name: scale_factor, dtype: object